# Análise de Redes Sociais com Métricas de Grafos

**Disciplina:** Teoria e Aplicação de Grafos (TAG) — CIC0199 - Turma 01 - 2026/2

**Docente:** Prof. Díbio Leandro Borges

**Integrantes:**
- Matheus Duarte da Silva - 211062277
- Suyanne Sara Miranda Silva — 222006445 

## Objetivo

Analisar as estruturas de redes sociais usando de métricas para grafos, avaliando a importância e potencial de difusão de informação de usuários em uma amostra real do Facebook


## Coleta dos Dados

### Configuração de bibliotecas e caminhos


In [ ]:
import colorsys
import random
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
from pyvis.network import Network

In [ ]:
# Configuração de diretórios e caminhos
DATASET_PATH = Path("../data/facebook_combined.txt.gz")

OUTPUT_METRICS_DIR = Path("../metrics")
OUTPUT_HTML_DIR = Path("../metrics/html")

OUTPUT_METRICS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_HTML_DIR.mkdir(parents=True, exist_ok=True)

# Sementes e tamanho da amostra
RANDOM_SEED = 17919131
SAMPLE_SIZE = 404

### Carrega a rede completa do Facebook


In [ ]:
# Carrega todo o grafo do Facebook
G = nx.read_edgelist(DATASET_PATH, nodetype=int)

print("Grafo completo carregada")
print(f"Total de nós (usuários): {G.number_of_nodes()}")
print(f"Total de arestas (conexões de amizade): {G.number_of_edges()}")

### Amostragem de 404 vértices


In [ ]:
# Gerador pseudo-aleatório com semente fixa para garantir reprodutibilidade
rng = random.Random(RANDOM_SEED)

# Seleciona 404 nós aleatórios
selected_nodes = rng.sample(list(G.nodes()), SAMPLE_SIZE)

print(f"Quantidade de nós selecionados: {len(selected_nodes)}")
print("Amostra dos 10 primeiros nós selecionados:", selected_nodes[:10])

## Construção do grafo

### Representa a rede amostrada


In [ ]:
# Pega o subgrafo e mantém os 404 nós e todas as arestas existentes entre eles
subgraph_sample = G.subgraph(selected_nodes).copy()

node_count = subgraph_sample.number_of_nodes()
edge_count = subgraph_sample.number_of_edges()
component_count = nx.number_connected_components(subgraph_sample)
isolated_node_count = len(list(nx.isolates(subgraph_sample)))

print(f"Nós no subgrafo amostrado: {node_count}")
print(f"Arestas no subgrafo amostrado: {edge_count}")
print(f"Número de componentes conexos: {component_count}")
print(f"Nós isolados (grau 0): {isolated_node_count}")

### Visualização inicial do subgrafo amostrado


In [ ]:
plt.figure(figsize=(12, 12))

nodes_positions = nx.spring_layout(
    subgraph_sample, seed=RANDOM_SEED, k=0.15, iterations=50
)

nx.draw(
    subgraph_sample,
    nodes_positions,
    node_size=40,
    node_color="skyblue",
    edge_color="gray",
    alpha=0.7,
    with_labels=False,
)

plt.title("Visualização Inicial do Subgrafo Amostrado (404 nós)", fontsize=14)

plt.tight_layout()
plt.axis("off")

plt.show()

## Extração de métricas do grafo

### Mapea as comunidades pelo algoritmo de Louvain


In [ ]:
# Identifica grupos de nós que têm mais conexões entre si do que com o resto da rede
communities = nx.community.louvain_communities(subgraph_sample, seed=RANDOM_SEED)

# Dicionário mapeando cada nó para o índice da sua comunidade
community_by_node = {
    node: i for i, community in enumerate(communities) for node in community
}

community_sizes = sorted([len(c) for c in communities], reverse=True)

print(f"Quantidade de comunidades identificadas: {len(communities)}")
print("Tamanho das 5 maiores comunidades:", community_sizes[:5])

### Visualização das comunidades


In [ ]:
# Mapa em cores das comunidades de Louvain
community_colors = [community_by_node[node] for node in subgraph_sample.nodes()]

plt.figure(figsize=(14, 10))
nx.draw_networkx_edges(subgraph_sample, nodes_positions, alpha=0.15, edge_color="gray")

nx.draw_networkx_nodes(
    subgraph_sample,
    nodes_positions,
    node_color=community_colors,
    cmap=plt.cm.tab20,
    node_size=40,
)

plt.title(
    f"Comunidades identificadas pelo algoritmo de Louvain (Total: {len(communities)})"
)

plt.tight_layout()
plt.axis("off")

plt.show()

### Calculando as medidas de centralidade

Calcula as métricas de centralidade para cada nó do subgrafo amostrado:

- **Grau de centralidade (Degree Centrality):** número de conexões normalizado pelo número total de outros nós.
- **Centralidade de intermediação (Betweenness Centrality):** frequência com que um nó se encontra nos caminhos mais curtos entre outros pares (pontes/gatekeepers).
- **Centralidade de proximidade (Closeness Centrality):** proximidade média aos nós alcançáveis (velocidade de propagação).
- **Centralidade de autovetor (Eigenvector Centrality):** influência proporcional à relevância dos próprios vizinhos conectados (prestígio).


In [ ]:
# Grau: quantas conexões diretas um nó possui, normalizadas pelo tamanho do grafo
nodes_degrees = nx.degree_centrality(subgraph_sample)

# Intermediação: quanto um nó participa dos caminhos mínimos entre outros nós
nodes_betweenness = nx.betweenness_centrality(subgraph_sample)

# Proximidade: quão perto um nó está dos demais, considerando os caminhos mínimos
nodes_closeness = nx.closeness_centrality(subgraph_sample)

# Centralidade de Autovetor: calcula a importância com base na importância dos vizinhos
eigenvector_centrality_scores = nx.eigenvector_centrality(
    subgraph_sample, max_iter=1000
)

print("Cálculo de centralidades:")
print(
    "Top 3 - Grau:", sorted(nodes_degrees.items(), key=lambda x: x[1], reverse=True)[:3]
)
print(
    "Top 3 - Intermediação:",
    sorted(nodes_betweenness.items(), key=lambda x: x[1], reverse=True)[:3],
)
print(
    "Top 3 - Proximidade:",
    sorted(nodes_closeness.items(), key=lambda x: x[1], reverse=True)[:3],
)
print(
    "Top 3 - Autovetor:",
    sorted(eigenvector_centrality_scores.items(), key=lambda x: x[1], reverse=True)[:3],
)

### Atribuindo as métricas aos vértices do grafo


In [ ]:
# Adiciona as métricas como atributos dos vértices no grafo
nx.set_node_attributes(
    G=subgraph_sample, values=community_by_node, name="Comunidades de Nós"
)
nx.set_node_attributes(
    G=subgraph_sample, values=nodes_degrees, name="Grau de Centralidade"
)
nx.set_node_attributes(
    G=subgraph_sample, values=nodes_betweenness, name="Centralidade de Intermediação"
)
nx.set_node_attributes(
    G=subgraph_sample, values=nodes_closeness, name="Centralidade de Proximidade"
)
nx.set_node_attributes(
    G=subgraph_sample,
    values=eigenvector_centrality_scores,
    name="Centralidade de Autovetor",
)

### Exportando os dados

In [ ]:
# Exporta o grafo completo para o Gephi (.gexf)
gexf_path = OUTPUT_METRICS_DIR / "graph_metrics.gexf"

nx.write_gexf(subgraph_sample, gexf_path)

print(f"Grafo exportado para o Gephi: {gexf_path}")

In [ ]:
# Consolida todas as métricas em formato DataFrame com Pandas
df_metrics = pd.DataFrame(
    {
        "no": list(subgraph_sample.nodes()),
        "comunidade": [community_by_node[no] for no in subgraph_sample.nodes()],
        "grau": [nodes_degrees[no] for no in subgraph_sample.nodes()],
        "intermediacao": [nodes_betweenness[no] for no in subgraph_sample.nodes()],
        "proximidade": [nodes_closeness[no] for no in subgraph_sample.nodes()],
        "autovetor": [
            eigenvector_centrality_scores[no] for no in subgraph_sample.nodes()
        ],
    }
)

# Exporta para CSV
csv_path = OUTPUT_METRICS_DIR / "graph_metrics.csv"
df_metrics.to_csv(csv_path, index=False)

print(f"Métricas salvas em CSV: {csv_path}")

df_metrics.head(10)

## Vizualização das métricas das métricas extraídas

### Grafos estáticos


In [ ]:
# Função auxiliar para visualizar as medidas de centralidade destacando os vértices com maiores valores
def display_centrality(
    G: nx.Graph, pos: dict, metricas: dict, nome_metrica: str, top_k=10
):

    top_k_nodes = sorted(metricas.items(), key=lambda item: item[1], reverse=True)[
        :top_k
    ]

    centrality_values = [metricas[no] for no in G.nodes()]
    top_nodes = [no for no, valor in top_k_nodes]

    plt.figure(figsize=(16, 12))

    nx.draw_networkx_edges(G, pos, alpha=0.08, edge_color="gray")

    node_visualization = nx.draw_networkx_nodes(
        G, pos, node_color=centrality_values, cmap=plt.cm.plasma, node_size=25
    )

    # Desenha os nós mais importantes com destaque em vermelho e tamanho maior
    nx.draw_networkx_nodes(
        G, pos, nodelist=top_nodes, node_color="red", node_size=180, edgecolors="black"
    )

    node_labels = {no: f"{no}\n{metricas[no]:.4f}" for no in top_nodes}

    nx.draw_networkx_labels(
        G,
        pos,
        labels=node_labels,
        font_size=9,
        font_weight="bold",
        bbox={"facecolor": "white", "edgecolor": "gray", "alpha": 0.9, "pad": 2},
    )

    plt.colorbar(node_visualization, label=f"Centralidade: {nome_metrica}")
    plt.title(f"{nome_metrica} — Top {top_k} nós", fontsize=14)

    plt.axis("off")
    plt.tight_layout()

    plt.show()

    return top_k_nodes

In [ ]:
top_degree_centrality = display_centrality(
    G=subgraph_sample, pos=nodes_positions, metricas=nodes_degrees, nome_metrica="Grau"
)

In [ ]:
top_intermediate_centrality = display_centrality(
    G=subgraph_sample,
    pos=nodes_positions,
    metricas=nodes_betweenness,
    nome_metrica="Intermediação",
)

In [ ]:
top_closeness = display_centrality(
    G=subgraph_sample,
    pos=nodes_positions,
    metricas=nodes_closeness,
    nome_metrica="Proximidade",
)

In [ ]:
top_eigenvector = display_centrality(
    G=subgraph_sample,
    pos=nodes_positions,
    metricas=eigenvector_centrality_scores,
    nome_metrica="Autovetor",
)

## Visualização interativa no navegador com Pyvis (HTML)

Recursos dos grafos interativos:

- Os nós em vermelho são os 10 maiores valores da métrica selecionada;
- O rótulo desses nós apresenta o ID e o valor da métrica;
- Os demais nós recebem cores gradientes conforme o valor e mostram seus IDs.


In [ ]:
# Gera um grafo interativo
interactive_graph = Network(
    height="100vh",
    width="100%",
    bgcolor="white",
    font_color=True,
    cdn_resources="in_line",
)

interactive_graph.from_nx(subgraph_sample)

html_export_path = OUTPUT_HTML_DIR / "generated_graph.html"
interactive_graph.write_html(str(html_export_path))

print(f"Grafo interativo base salvo em: {html_export_path}")

In [ ]:
# Função auxiliar para gerar visualizações interativas em HTML para cada métrica
def render_visualization_metrics(
    G: nx.Graph, metricas: dict, nome_metrica: str, top_k=10
):

    highest_metrics = sorted(metricas.items(), key=lambda item: item[1], reverse=True)[
        :top_k
    ]
    top_metric_nodes = {no for no, valor in highest_metrics}

    lowest_metric = min(metricas.values())
    highest_metric = max(metricas.values())

    graph_representation = Network(
        height="100vh",
        width="100%",
        bgcolor="white",
        font_color=True,
        cdn_resources="in_line",
    )

    for node in G.nodes():
        value = metricas[node]
        proportion_value = (
            (value - lowest_metric) / (highest_metric - lowest_metric)
            if highest_metric > lowest_metric
            else 0.5
        )

        color: tuple | str = colorsys.hsv_to_rgb(
            0.7 * (1 - proportion_value), 0.85, 0.95
        )
        color = f"#{int(color[0] * 255):02x}{int(color[1] * 255):02x}{int(color[2] * 255):02x}"

        is_top_metric_node = node in top_metric_nodes
        node_label = f"{node}\n{value:.4f}" if is_top_metric_node else str(node)

        graph_representation.add_node(
            node,
            label=node_label,
            title=f"Nó: {node}<br>{nome_metrica}: {value:.6f}",
            color="red" if is_top_metric_node else color,
            size=28 if is_top_metric_node else 10,
            font={"size": 16 if is_top_metric_node else 10},
        )

    graph_representation.add_edges(G.edges())

    file_name = (
        nome_metrica.lower()
        .replace(" ", "_")
        .replace("ã", "a")
        .replace("ç", "c")
        .replace("é", "e")
    )

    html_file_path = OUTPUT_HTML_DIR / f"{file_name}.html"

    graph_representation.write_html(str(html_file_path))

    print(f"Arquivo HTML gerado: {html_file_path}")

    return highest_metrics


# Função auxiliar para gerar visualização interativa
def render_communities(G: nx.Graph, community_dict: dict):

    visualization_graph = Network(
        height="100vh",
        width="100%",
        bgcolor="white",
        font_color=True,
        cdn_resources="in_line",
    )

    for node in G.nodes():
        community_id = community_dict[node]

        rgb_value = colorsys.hsv_to_rgb((community_id % 20) / 20.0, 0.75, 0.9)
        hex_color = f"#{int(rgb_value[0] * 255):02x}{int(rgb_value[1] * 255):02x}{int(rgb_value[2] * 255):02x}"

        visualization_graph.add_node(
            node,
            label=str(node),
            title=f"Nó: {node}<br>Comunidade: {community_id}",
            color=hex_color,
            size=12,
        )

    visualization_graph.add_edges(G.edges())

    output_file_path = OUTPUT_HTML_DIR / "comunidades_louvain.html"
    visualization_graph.write_html(str(output_file_path))

    print(f"Arquivo de comunidades gerado: {output_file_path}")

In [ ]:
# Gera os arquivos HTML para visualização individual e integração no analise_grafo.html
render_visualization_metrics(subgraph_sample, nodes_degrees, "Grau")
render_visualization_metrics(subgraph_sample, nodes_betweenness, "Intermediacao")
render_visualization_metrics(subgraph_sample, nodes_closeness, "Proximidade")
render_visualization_metrics(
    subgraph_sample, eigenvector_centrality_scores, "Autovetor"
)

render_communities(subgraph_sample, community_by_node)

# 5. Relatório e Análise do Grafo
### 5.1 Síntese Comparativa dos Nós de Maior Influência


In [ ]:
# Tabela comparativa dos Top 5 nós em cada uma das métricas calculadas
top_k = 5

top_summary = pd.DataFrame(
    {
        "Top Grau": [
            node
            for node, _ in sorted(
                nodes_degrees.items(), key=lambda x: x[1], reverse=True
            )[:top_k]
        ],
        "Val. Grau": [
            round(val, 4)
            for _, val in sorted(
                nodes_degrees.items(), key=lambda x: x[1], reverse=True
            )[:top_k]
        ],
        "Top Intermediação": [
            no
            for no, _ in sorted(
                nodes_betweenness.items(), key=lambda x: x[1], reverse=True
            )[:top_k]
        ],
        "Val. Intermediação": [
            round(val, 4)
            for _, val in sorted(
                nodes_betweenness.items(), key=lambda x: x[1], reverse=True
            )[:top_k]
        ],
        "Top Proximidade": [
            no
            for no, _ in sorted(
                nodes_closeness.items(), key=lambda x: x[1], reverse=True
            )[:top_k]
        ],
        "Val. Proximidade": [
            round(val, 4)
            for _, val in sorted(
                nodes_closeness.items(), key=lambda x: x[1], reverse=True
            )[:top_k]
        ],
        "Top Autovetor": [
            no
            for no, _ in sorted(
                eigenvector_centrality_scores.items(), key=lambda x: x[1], reverse=True
            )[:top_k]
        ],
        "Val. Autovetor": [
            round(val, 4)
            for _, val in sorted(
                eigenvector_centrality_scores.items(), key=lambda x: x[1], reverse=True
            )[:top_k]
        ],
    }
)

top_summary

### 5.2 Interpretação Sociológica e Estrutural dos Resultados

1. **Grau de Centralidade (Popularidade Local):**
   - **Líderes:** O nó **1447** possui o maior grau na rede amostrada, seguido de perto por **1175** e **1598**.
   - **Interpretação:** Representam os membros com mais conexões imediatas de amizade na amostra. São os centros locais de atenção e popularidade direta.

2. **Centralidade de Intermediação (Pontes e Gatekeepers):**
   - **Líderes:** Os nós **1537**, **1085** e **1029** dominam o betweenness.
   - **Insight Crítico:** O nó **1537** não é o primeiro em grau, mas atua como a ponte estratégica fundamental entre clusters distintos. Esses nós controlam o fluxo de informações; sua desconexão causaria a separação de grupos inteiros na rede.

3. **Centralidade de Proximidade (Velocidade e Eficiência de Difusão):**
   - **Líderes:** O nó **1029** lidera com proximidade destacada, seguido por **1175**, **1447** e **1537**.
   - **Interpretação:** Mede quem tem o menor caminho médio até todos os outros nós acessíveis. Usuários com alta proximidade são os melhores candidatos para **iniciar campanhas de difusão de informação**, pois a mensagem atinge a rede em menos saltos.

4. **Centralidade de Autovetor (Influência Sistêmica e Prestígio):**
   - **Líderes:** Os nós **1447**, **1598** e **1175** concentram os maiores escores de autovetor.
   - **Interpretação:** Reflete estar conectado a outros nós que também são altamente conectados. Esses usuários formam o núcleo de prestígio social da rede.

5. **Topologia da Amostragem e Fragmentação:**
   - A extração uniforme de 404 nós (~10% da rede SNAP de 4039 nós) gera fragmentação estrutural, resultando em múltiplos componentes conexos e nós isolados.
   - Isso explica a presença de valores nulos para intermediação em nós periféricos ou isolados, evidenciando o comportamento característico de redes de mundo pequeno sob amostragem uniforme.

6. **Exploração no Dashboard Unificado:**
   - Todas as visualizações interativas geradas (`grau.html`, `intermediacao.html`, `proximidade.html`, `autovetor.html` e `comunidades_louvain.html`) podem ser visualizadas conjuntamente em abas abrindo o arquivo `metrics/html/analise_grafo.html` diretamente no navegador.
